### Creates `openalex.works.crossref_super_authorships` in Walden End to End workflow

In [ ]:
CREATE OR REPLACE TABLE openalex.works.crossref_super_authorships AS
WITH -- oxjob #1790: Crossref peer reviews (F1000Research, Gates / HRB / Wellcome Open and NIHR Open reviewer reports,
-- Copernicus referee comments, eLife public reviews, SciPost reports, Cambridge Prisms reviews ...). Their landing page
-- describes the REVIEWED article, so its citation_author tags are that article's authors, never the reviewer's: 84 of
-- 84 sampled reviews showed the reviewed article's authors or junk (28 Oct 2026). Crossref names the reviewer when one
-- is named; the landing page never gives a peer review its author list. crossref_base below keeps landing-page rows,
-- so a review whose Crossref authors are the article's own (Copernicus author comments) still gets their affiliations.
-- Revert: drop this CTE, its LEFT JOIN and the "pr.doi IS NULL" line in crossref_authoritative_base.
crossref_peer_review_dois AS (
  SELECT DISTINCT native_id AS doi
  FROM openalex.works.locations_parsed
  WHERE provenance = 'crossref'
    AND (raw_type = 'peer-review' OR type = 'peer-review')
),

-- oxjob #2274: IUCN Red List assessments (10.2305/IUCN.UK.<year>.RLTS.T...). Crossref names one corporate author,
-- 'IUCN', on ~255K of them, while the assessment page's citation_author tags name the assessors the Red List cites
-- (1-11 people, or the assessing body: 'BirdLife International', 'Mollusc Specialist Group'). On such a DOI Crossref's
-- lone 'IUCN' never beats the page's list when that list is a clean byline: every name has a letter, none is an e-mail
-- address (a placeholder account, 'false.email@globaltrees.org'), none holds several people in one string ('Lovett, J. &
-- Clarke, G.P.', 'Esser, L. and Cumberlidge, N.', 'A, B, C, D'; a body's name may hold '&' or 'and'), and none still
-- carries the assessor's affiliation in parentheses ('Alan Williams (CSIRO Tasmania, Australia)'), as a page parsed
-- before parseland-lib's iucn_red_list parser does (that parser moves it to the affiliation, so a row with affiliations
-- has been through it). Same order as crossref_authoritative_base, so the row tested is the row that wins. Without a
-- clean page list the DOI keeps Crossref's 'IUCN'. Revert: drop this CTE, its LEFT JOIN and the "iu.doi IS NULL" lines in
-- crossref_authoritative_base.
iucn_red_list_page_bylines AS (
  SELECT doi
  FROM (
    SELECT
      get(filter(ids, x -> x.namespace = "doi").id, 0) AS doi,
      authors,
      affiliations_exist,
      ROW_NUMBER() OVER (
        PARTITION BY get(filter(ids, x -> x.namespace = "doi").id, 0)
        ORDER BY updated_date DESC NULLS LAST, native_id ASC, xxhash64(to_json(authors)) ASC
      ) AS r
    FROM openalex.works.locations_parsed
    WHERE provenance = 'landing_page'
      AND authors_exist
      AND get(filter(ids, x -> x.namespace = "doi").id, 0) LIKE '10.2305/%'
  )
  WHERE r = 1
    AND lower(doi) LIKE '%.rlts.%'
    AND SIZE(authors) > 0
    AND FORALL(authors, a -> COALESCE(
          a.name RLIKE '\\p{L}'
          AND NOT a.name LIKE '%@%'
          AND (NOT a.name RLIKE '(?i)&|;|\\s(and|y|e|et|und|en)\\s'
               OR a.name RLIKE '(?i)\\b(group|team|commission|council|society|network|association|institute|university|centre|center|foundation|programme|program)\\b')
          AND SIZE(split(a.name, ',')) <= 2
          AND (NOT a.name LIKE '%(%' OR affiliations_exist)
          AND lower(trim(a.name)) <> 'iucn', FALSE))
),

-- Authoritative sources for author names (crossref and landing_page, NOT pdf)
crossref_authoritative_base AS (
  SELECT
    CASE
      WHEN provenance = 'crossref' THEN native_id
      ELSE get(filter(ids, x -> x.namespace = "doi").id, 0)
    END AS doi,
    authors,
    priority,
    ROW_NUMBER() OVER (
      PARTITION BY
        CASE
          WHEN provenance = 'crossref' THEN native_id
          ELSE get(filter(ids, x -> x.namespace = "doi").id, 0)
        END
      ORDER BY priority ASC, updated_date DESC NULLS LAST, native_id ASC, xxhash64(to_json(authors)) ASC
    ) AS r
  FROM openalex.works.locations_parsed lp
    LEFT JOIN openalex.system.priority_table USING (provenance)
    LEFT JOIN crossref_peer_review_dois pr
      ON lp.provenance = 'landing_page' AND pr.doi = get(filter(lp.ids, x -> x.namespace = "doi").id, 0)
    LEFT JOIN iucn_red_list_page_bylines iu ON lp.provenance = 'crossref' AND iu.doi = lp.native_id  -- oxjob #2274
  WHERE authors_exist
    AND provenance IN ('crossref', 'landing_page')
    AND pr.doi IS NULL  -- oxjob #1790: no landing-page authors for a peer review
    -- oxjob #2108: a Crossref list that is only the Ovid / Adis 'not available' marker ('&NA;', '&NA; &NA;') names no
    -- author, so it never beats a landing-page list for the same DOI (~6K Wolters Kluwer items, mostly 2006
    -- Transplantation congress abstracts and signed letters and reviews, whose page prints the names). With no
    -- landing-page list the location keeps its marker and CreateLocationsMapped drops it (same test there).
    -- Revert: delete these lines.
    AND NOT (provenance = 'crossref' AND SIZE(authors) > 0 AND FORALL(authors, a -> COALESCE(
          a.name ILIKE '%na;%' AND a.name RLIKE '(?i)&(amp;)*na;'
          AND NOT regexp_replace(a.name, '(?i)&(amp;)*na;', '') RLIKE '[\\p{L}\\p{N}]', FALSE)))
    -- oxjob #2274: Crossref's lone 'IUCN' never beats a clean Red List page byline (iucn_red_list_page_bylines)
    AND NOT (iu.doi IS NOT NULL AND SIZE(authors) = 1 AND COALESCE(lower(trim(authors[0].name)), '') = 'iucn')
),

-- oxjob #1062: APS (10.1103) DOIs with a landing-page row that has affiliations and lists exactly Crossref's author
-- keys in Crossref's order (see crossref_base.priority).
aps_page_lined_up AS (
  SELECT DISTINCT cr.native_id AS doi
  FROM openalex.works.locations_parsed cr
    INNER JOIN openalex.works.locations_parsed pg
      ON pg.provenance = 'landing_page' AND pg.affiliations_exist
      AND get(filter(pg.ids, x -> x.namespace = "doi").id, 0) = cr.native_id
  WHERE cr.provenance = 'crossref'
    AND cr.native_id LIKE '10.1103/%'
    AND get(filter(pg.ids, x -> x.namespace = "doi").id, 0) LIKE '10.1103/%'
    AND SIZE(cr.authors) > 0
    AND transform(pg.authors, a -> a.author_key) = transform(cr.authors, a -> a.author_key)
    -- a page line that lost its linked name (", Shanghai 201210, China" for "ShanghaiTech Laboratory for Topological
    -- Physics, Shanghai ...") keeps the whole DOI on Crossref
    AND NOT EXISTS(pg.authors, a -> EXISTS(a.affiliations, f -> COALESCE(f.name RLIKE '^\\s*[,;]', FALSE)))
),

-- Sources for affiliations and is_corresponding (crossref, pubmed, pdf, and landing_page)
crossref_base AS (
  SELECT
    CASE
      WHEN provenance = 'crossref' THEN native_id
      ELSE get(filter(ids, x -> x.namespace = "doi").id, 0)
    END AS doi,
    authors,
    provenance,
    -- oxjob #1062 (ZD 3381): since 2024 APS (10.1103) deposits in Crossref only the ROR-anchored fragment of each
    -- affiliation line ("Sorbonne Université" for "Laboratoire Jacques-Louis Lions, Sorbonne Université, 4 Place
    -- Jussieu ..."; name-only strings 0.1% of 2023 APS strings, 79% of 2025's). Its page's citation_author_institution
    -- prints the whole line. On a 10.1103 DOI whose page lists exactly Crossref's authors in Crossref's order
    -- (aps_page_lined_up), Crossref drops just below the landing page (priority 8: after landing_page 7, before pdf 9),
    -- so an author the page gives affiliations takes the page's and an author it leaves empty keeps Crossref's; PubMed
    -- still outranks both. The lined-up test keeps the #726 tie-break working for co-authors who share a key (BESIII
    -- "wang;x"): a page whose list differs would hand each of them the union. The ROR anchors in the page's lines are
    -- stripped in the final SELECT. Revert: back to a bare `priority,` and drop aps_page_lined_up.
    CASE WHEN apl.doi IS NOT NULL THEN 8 ELSE priority END AS priority,
    affiliations_exist,
    is_corresponding_exists,
    ROW_NUMBER() OVER (
      PARTITION BY
        CASE
          WHEN provenance = 'crossref' THEN native_id
          ELSE get(filter(ids, x -> x.namespace = "doi").id, 0)
        END
      ORDER BY priority ASC, updated_date DESC NULLS LAST, native_id ASC, xxhash64(to_json(authors)) ASC
    ) AS r
  FROM openalex.works.locations_parsed
    LEFT JOIN openalex.system.priority_table USING (provenance)
    LEFT JOIN aps_page_lined_up apl ON provenance = 'crossref' AND apl.doi = native_id  -- oxjob #1062
  WHERE authors_exist
    AND provenance IN ('crossref', 'pubmed', 'pdf', 'landing_page')
),

crossref_best_authors_exploded AS (
  SELECT
    doi,
    SIZE(authors) AS author_count,
    posexplode(authors) AS (original_author_order, best_author_list_exploded),
    best_author_list_exploded.author_key AS author_key
  FROM crossref_authoritative_base
  WHERE r = 1
),

-- How many authors in the authoritative list carry each key (2+ = co-authors who share a key)
crossref_best_authors_keyed AS (
  SELECT
    *,
    COUNT(*) OVER (PARTITION BY doi, author_key) AS key_count
  FROM crossref_best_authors_exploded
),

-- Key-based affiliations (existing logic)
crossref_affiliations_base AS (
  SELECT doi, affiliations, author_key, priority
  FROM (
    SELECT
      doi,
      explode(authors) AS authors_exploded,
      authors_exploded.affiliations AS affiliations,
      authors_exploded.author_key AS author_key,
      priority,
      provenance
    FROM crossref_base
    WHERE affiliations_exist
  )
  -- Guard against GROBID/scraper noise: cap affiliations per author for parsed sources
  WHERE (
    provenance NOT IN ('pdf', 'landing_page')
    OR (provenance = 'pdf' AND SIZE(affiliations) <= 2)
    OR (provenance = 'landing_page' AND SIZE(affiliations) <= 5)
  )
),

crossref_affiliations_staging AS (
  SELECT
    *,
    explode(affiliations) AS exploded_affiliations,
    RANK() OVER (PARTITION BY doi, author_key ORDER BY priority ASC) AS r
  FROM crossref_affiliations_base
  WHERE get(affiliations.name, 0) IS NOT NULL
),

crossref_affiliations AS (
  SELECT
    doi,
    author_key,
    array_sort(collect_list(exploded_affiliations)) AS affiliations
  FROM crossref_affiliations_staging
  WHERE r = 1
  GROUP BY doi, author_key
),

-- oxjob #1250: an author whose best list starts with an id-only entry (Optica deposits {ROR, no name} first) lost the
-- named entries after it too: every staging filter above tests only the FIRST entry's name. These lists fill a seat
-- that every other source (key, tiebreak, position, PMC) leaves empty, and nothing else: last term of the COALESCE.
-- The id-only entries ride along and CreateWorksBase drops their NULL names; their ror_id stays unused (Jason, 30 Sep
-- 2026, #1467: publisher ROR ids never assign institutions). Revert: drop this CTE, its LEFT JOIN and the COALESCE tail.
crossref_named_after_id_only AS (
  SELECT doi, author_key, array_sort(collect_list(exploded_affiliations)) AS affiliations
  FROM (
    SELECT
      doi, author_key,
      explode(affiliations) AS exploded_affiliations,
      RANK() OVER (PARTITION BY doi, author_key ORDER BY priority ASC) AS r
    FROM crossref_affiliations_base
    WHERE get(affiliations.name, 0) IS NULL
      AND exists(affiliations, a -> a.name IS NOT NULL)
  )
  WHERE r = 1
  GROUP BY doi, author_key
),

-- oxjob #726: co-authors who share a key ("chakraborty;c" for Chinmay and Chandan Chakraborty, "mao;s" for Seabert and
-- Skyler Mao) each took the UNION of both their affiliations, because crossref_affiliations is one row per
-- (doi, author_key). When the co-authors have different names, each takes the affiliations at their own seat, read
-- from the same source(s) crossref_affiliations used for the key (its best priority), provided that source lists as
-- many authors and the same key at that seat (the tie-break crossref_is_corresponding_tiebreak uses for the flag). A
-- seat that source leaves empty stays empty. The same person listed twice (identical names) keeps the union, as does a
-- key whose source doesn't line up. Revert: drop these three CTEs, their two LEFT JOINs and the CASE line in the
-- affiliations COALESCE below.
crossref_affiliations_tiebreak_base AS (
  SELECT
    *,
    -- the priority crossref_affiliations takes this key's affiliations from
    MIN(CASE WHEN get(affiliations.name, 0) IS NOT NULL THEN priority END) OVER (PARTITION BY doi, author_key) AS key_best_priority
  FROM (
    SELECT
      doi,
      source_author_count,
      author_position,
      author_exploded.author_key AS author_key,
      author_exploded.affiliations AS affiliations,
      priority,
      provenance
    FROM (
      SELECT
        doi,
        SIZE(authors) AS source_author_count,
        priority,
        provenance,
        posexplode(authors) AS (author_position, author_exploded)
      FROM crossref_base
      WHERE affiliations_exist
        AND doi IN (SELECT doi FROM crossref_best_authors_keyed WHERE key_count > 1)
    )
  )
  -- the same caps as crossref_affiliations_base
  WHERE (
    provenance NOT IN ('pdf', 'landing_page')
    OR (provenance = 'pdf' AND SIZE(affiliations) <= 2)
    OR (provenance = 'landing_page' AND SIZE(affiliations) <= 5)
  )
),

-- Shared keys held by co-authors with different names whose best source lines up with the authoritative list
crossref_affiliations_tiebreak_keys AS (
  SELECT DISTINCT ba.doi, ba.author_key
  FROM (
    SELECT
      doi, author_key, original_author_order, author_count,
      SIZE(COLLECT_SET(LOWER(CONCAT_WS('|', TRIM(best_author_list_exploded.given), TRIM(best_author_list_exploded.family),
        TRIM(best_author_list_exploded.name)))) OVER (PARTITION BY doi, author_key)) AS key_names
    FROM crossref_best_authors_keyed
    WHERE key_count > 1
  ) ba
    INNER JOIN crossref_affiliations_tiebreak_base t
      ON t.doi = ba.doi AND t.author_position = ba.original_author_order
      AND t.author_key = ba.author_key AND t.source_author_count = ba.author_count
      AND t.priority = t.key_best_priority
  WHERE ba.key_names > 1
    AND get(t.affiliations.name, 0) IS NOT NULL
),

crossref_affiliations_tiebreak AS (
  SELECT
    doi,
    author_position,
    author_key,
    source_author_count,
    array_sort(collect_list(exploded_affiliations)) AS affiliations
  FROM (
    SELECT
      doi, author_position, author_key, source_author_count,
      explode(affiliations) AS exploded_affiliations
    FROM crossref_affiliations_tiebreak_base
    WHERE priority = key_best_priority
      AND get(affiliations.name, 0) IS NOT NULL
  )
  GROUP BY doi, author_position, author_key, source_author_count
),

-- Positional fallback: affiliations by author position instead of key.
-- Used when author_key mismatch prevents key-based matching (e.g. CJK vs romanized names).
crossref_affiliations_positional_base AS (
  SELECT doi, affiliations, author_position, priority
  FROM (
    SELECT
      doi,
      posexplode(authors) AS (author_position, author_exploded),
      author_exploded.affiliations AS affiliations,
      priority,
      provenance
    FROM crossref_base
    WHERE affiliations_exist
  )
  -- Guard against GROBID/scraper noise: cap affiliations per author for parsed sources
  WHERE (
    provenance NOT IN ('pdf', 'landing_page')
    OR (provenance = 'pdf' AND SIZE(affiliations) <= 2)
    OR (provenance = 'landing_page' AND SIZE(affiliations) <= 5)
  )
),

crossref_affiliations_positional_staging AS (
  SELECT
    doi, author_position, priority,
    explode(affiliations) AS exploded_affiliations,
    RANK() OVER (PARTITION BY doi, author_position ORDER BY priority ASC) AS r
  FROM crossref_affiliations_positional_base
  WHERE get(affiliations.name, 0) IS NOT NULL
),

crossref_affiliations_positional AS (
  SELECT
    doi,
    author_position,
    array_sort(collect_list(exploded_affiliations)) AS affiliations
  FROM crossref_affiliations_positional_staging
  WHERE r = 1
  GROUP BY doi, author_position
),

-- Every non-null is_corresponding flag from a non-PDF source, with its position and its source's author count.
-- pdf excluded until the #736 merge policy ships.
crossref_is_corresponding_flags AS (
  SELECT
    doi,
    source_author_count,
    author_position,
    author_exploded.author_key AS author_key,
    author_exploded.is_corresponding AS flag
  FROM (
    SELECT
      doi,
      SIZE(authors) AS source_author_count,
      posexplode(authors) AS (author_position, author_exploded)
    FROM crossref_base
    WHERE is_corresponding_exists
      AND provenance != 'pdf'
  )
  WHERE author_exploded.is_corresponding IS NOT NULL
),

-- Key-based is_corresponding: ONE row per (doi, author_key), so the join below can't fan out (oxjob #1805). It used to
-- keep one row per flag: when two sources disagree on an author, or two co-authors share a key ("li;j" for Junhong Li
-- and Jun Li) and a page flags one of them, each author joined both rows and COLLECT_SET kept a true and a false copy,
-- so the work listed those authors twice (301,801 DOIs, 26 Oct 2026). Any source flagging the author wins, as the
-- true copy did before. Co-authors who share a key and are flagged both ways are settled by position first
-- (tie-break below).
crossref_is_corresponding AS (
  SELECT
    doi,
    author_key,
    MAX(flag) AS is_corresponding_landing_page,
    MIN(flag) = MAX(flag) AS flags_agree
  FROM crossref_is_corresponding_flags
  GROUP BY doi, author_key
),

-- Tie-break for co-authors who share a key in the authoritative list: the flag at the author's own position, from
-- sources that list as many authors and the same key there. One row per join key, so no fan-out either.
crossref_is_corresponding_tiebreak AS (
  SELECT
    doi,
    author_position,
    author_key,
    source_author_count,
    MAX(flag) AS is_corresponding_landing_page
  FROM crossref_is_corresponding_flags
  GROUP BY doi, author_position, author_key, source_author_count
),

-- Positional fallback: is_corresponding by position, one row per position
crossref_is_corresponding_positional AS (
  SELECT
    doi,
    author_position,
    MAX(flag) AS is_corresponding_landing_page
  FROM crossref_is_corresponding_flags
  GROUP BY doi, author_position
),

-- oxjob #2223: a page that prints its byline in another script (J-STAGE's Japanese bylines) flags its corresponding
-- author under a key no Crossref author has ("中島;駿" vs "nakajima;s"), so the flag never reached the work: the
-- positional fallback above runs only when NO affiliation matched by key, and Crossref's own affiliations always do
-- (50,919 visible works, 50,371 of them J-STAGE, 13 Nov 2026). A source's flags are taken by position when (1) a name
-- it flags is in Han, kana, Hangul or Cyrillic script, (2) it shares no author key with the authoritative list,
-- (3) it lists as many authors, and (4) no source flags anyone on the DOI by key. 33 of 33 sampled J-STAGE works line
-- up seat for seat. Latin-script mismatches are left alone (degrees in names, swapped order, every author starred).
-- Revert: drop these two CTEs, their LEFT JOIN and the xsc line in the COALESCE below.
cross_script_corresponding_sources AS (
  SELECT c.doi, c.authors
  FROM crossref_base c
    INNER JOIN crossref_authoritative_base auth ON auth.doi = c.doi AND auth.r = 1
  WHERE c.provenance != 'pdf'
    AND c.is_corresponding_exists
    AND EXISTS(c.authors, x -> COALESCE(x.is_corresponding, FALSE)
      AND COALESCE(x.name RLIKE '\\p{IsHan}|\\p{IsHiragana}|\\p{IsKatakana}|\\p{IsHangul}|\\p{IsCyrillic}', FALSE))
    AND SIZE(c.authors) = SIZE(auth.authors)
    AND NOT EXISTS(c.authors, x -> COALESCE(ARRAY_CONTAINS(TRANSFORM(auth.authors, y -> y.author_key), x.author_key), FALSE))
),

cross_script_corresponding AS (
  SELECT s.doi, s.author_position, TRUE AS is_corresponding_landing_page
  FROM (
    SELECT doi, posexplode(authors) AS (author_position, author_exploded)
    FROM cross_script_corresponding_sources
  ) s
    LEFT ANTI JOIN (
      -- DOIs where some source already flags an author of the authoritative list by key: left as they are
      SELECT DISTINCT ba.doi
      FROM crossref_best_authors_exploded ba
        INNER JOIN crossref_is_corresponding cor ON cor.doi = ba.doi AND cor.author_key = ba.author_key
      WHERE cor.is_corresponding_landing_page
    ) keyed ON keyed.doi = s.doi
  WHERE COALESCE(s.author_exploded.is_corresponding, FALSE)
  GROUP BY s.doi, s.author_position
),

-- oxjob #2274: a page that prints the same byline with noise in its names ("S. Stock 1", "Ivan K. Crosby, MBBS",
-- "Eiichi Nakamura Prof. Dr.", "Hongfu Li SciProfiles Scilit", "Abd El Rahman, Yasmin") keys its starred author
-- differently from Crossref ("1;s" vs "stock;s", "prof;e" vs "nakamura;e"), so the flag never reached the work: ~176K
-- visible works showed no corresponding author on night 2026-11-15 (#2274 EXPLORE § 1). A source's flags are taken by
-- position when (1) it lists as many authors as the authoritative list, (2) it does not flag every author of a
-- multi-author list (junk stars), (3) at every seat it flags, the authoritative author's key surname is a whole word of
-- the source's name there, or the source's key surname a whole word of the authoritative name (lower case, accents
-- folded as the key folds them, hyphens, apostrophes and dots dropped, other letters dropped), and (4) no source flags
-- anyone on the DOI by key (as for cross_script_corresponding). Revert: drop these three CTEs, their LEFT JOIN and the
-- nnc line in the COALESCE below.
corresponding_keyed_dois AS (
  SELECT DISTINCT ba.doi
  FROM crossref_best_authors_exploded ba
    INNER JOIN crossref_is_corresponding cor ON cor.doi = ba.doi AND cor.author_key = ba.author_key
  WHERE cor.is_corresponding_landing_page
),

name_noise_corresponding_seats AS (
  SELECT
    c.doi,
    TRANSFORM(c.authors, (x, i) -> NAMED_STRUCT(
      'author_position', i,
      'flag', COALESCE(x.is_corresponding, FALSE),
      'same_person', COALESCE(
        (LENGTH(SPLIT(GET(auth.authors, i).author_key, ';')[0]) >= 2
          AND ARRAY_CONTAINS(SPLIT(TRIM(REGEXP_REPLACE(REGEXP_REPLACE(TRANSLATE(LOWER(COALESCE(x.name, CONCAT_WS(' ', x.given, x.family))),
            'àáâãäåçèéêëìíîïñòóôõöùúûüýÿāăąćĉċčďēĕėęěĝğġģĥĩīĭįĵķĺļľŀńņňŉōŏőŕŗřśŝşšţťũūŭůűųŵŷźżžſơưǎǐǒǔǖǘǚǜǟǡǧǩǫǭǰǵǹǻȁȃȅȇȉȋȍȏȑȓȕȗșțȟȧȩȫȭȯȱȳ',
            'aaaaaaceeeeiiiinooooouuuuyyaaaccccdeeeeegggghiiiijkllllnnnnooorrrssssttuuuuuuwyzzzsouaiouuuuuaagkoojgnaaaeeiioorruusthaeooooy'),
            '[-‐\'’.]|[\\p{L}&&[^a-z]]', ''), '[^a-z]+', ' ')), ' '),
            SPLIT(GET(auth.authors, i).author_key, ';')[0]))
        OR (LENGTH(SPLIT(x.author_key, ';')[0]) >= 2
          AND ARRAY_CONTAINS(SPLIT(TRIM(REGEXP_REPLACE(REGEXP_REPLACE(TRANSLATE(LOWER(CONCAT_WS(' ', GET(auth.authors, i).given, GET(auth.authors, i).family, GET(auth.authors, i).name)),
            'àáâãäåçèéêëìíîïñòóôõöùúûüýÿāăąćĉċčďēĕėęěĝğġģĥĩīĭįĵķĺļľŀńņňŉōŏőŕŗřśŝşšţťũūŭůűųŵŷźżžſơưǎǐǒǔǖǘǚǜǟǡǧǩǫǭǰǵǹǻȁȃȅȇȉȋȍȏȑȓȕȗșțȟȧȩȫȭȯȱȳ',
            'aaaaaaceeeeiiiinooooouuuuyyaaaccccdeeeeegggghiiiijkllllnnnnooorrrssssttuuuuuuwyzzzsouaiouuuuuaagkoojgnaaaeeiioorruusthaeooooy'),
            '[-‐\'’.]|[\\p{L}&&[^a-z]]', ''), '[^a-z]+', ' ')), ' '),
            SPLIT(x.author_key, ';')[0])),
        FALSE)
    )) AS seats
  FROM crossref_base c
    INNER JOIN crossref_authoritative_base auth ON auth.doi = c.doi AND auth.r = 1
  WHERE c.provenance != 'pdf'
    AND c.is_corresponding_exists
    AND SIZE(c.authors) = SIZE(auth.authors)
    AND NOT (SIZE(c.authors) > 1 AND FORALL(c.authors, x -> COALESCE(x.is_corresponding, FALSE)))
),

name_noise_corresponding AS (
  SELECT s.doi, s.seat.author_position AS author_position, TRUE AS is_corresponding_landing_page
  FROM (
    SELECT doi, EXPLODE(FILTER(seats, z -> z.flag)) AS seat
    FROM name_noise_corresponding_seats
    WHERE FORALL(seats, z -> NOT z.flag OR z.same_person)
  ) s
    LEFT ANTI JOIN corresponding_keyed_dois keyed ON keyed.doi = s.doi
  GROUP BY s.doi, s.seat.author_position
),

-- oxjob #2274 (fix 7): the same name noise keeps a page's affiliations off their seat. Pages that print a name
-- family-first ('Family Given' keyed 'given;f' against Crossref's 'family;g'), degrees, titles, footnote digits:
-- the key join misses that seat, and the positional fallback runs only when no seat on the DOI matched by key. A
-- landing-page list gives its affiliations by position to seats that nothing above fills when (1) it lists as many
-- authors as the authoritative list and (2) at EVERY seat the same-person test of name_noise_corresponding_seats holds
-- (the whole byline is the same people in the same order); at most 5 strings a seat, as for every page source.
-- Revert: drop these two CTEs, their LEFT JOIN and the nna line in the affiliations COALESCE below.
-- DOIs with a seat no key-matched affiliation reaches (the name test below is costly; most DOIs never need it)
name_noise_affiliation_dois AS (
  SELECT DISTINCT ba.doi
  FROM crossref_best_authors_exploded ba
    LEFT ANTI JOIN crossref_affiliations ca ON ca.doi = ba.doi AND ca.author_key = ba.author_key
),

name_noise_affiliation_seats AS (
  SELECT
    c.doi,
    c.r,
    TRANSFORM(c.authors, (x, i) -> NAMED_STRUCT(
      'author_position', i,
      'affiliations', IF(SIZE(x.affiliations) <= 5, FILTER(x.affiliations, f -> f.name IS NOT NULL), ARRAY()),
      'same_person', COALESCE(
        (LENGTH(SPLIT(GET(auth.authors, i).author_key, ';')[0]) >= 2
          AND ARRAY_CONTAINS(SPLIT(TRIM(REGEXP_REPLACE(REGEXP_REPLACE(TRANSLATE(LOWER(COALESCE(x.name, CONCAT_WS(' ', x.given, x.family))),
            'àáâãäåçèéêëìíîïñòóôõöùúûüýÿāăąćĉċčďēĕėęěĝğġģĥĩīĭįĵķĺļľŀńņňŉōŏőŕŗřśŝşšţťũūŭůűųŵŷźżžſơưǎǐǒǔǖǘǚǜǟǡǧǩǫǭǰǵǹǻȁȃȅȇȉȋȍȏȑȓȕȗșțȟȧȩȫȭȯȱȳ',
            'aaaaaaceeeeiiiinooooouuuuyyaaaccccdeeeeegggghiiiijkllllnnnnooorrrssssttuuuuuuwyzzzsouaiouuuuuaagkoojgnaaaeeiioorruusthaeooooy'),
            '[-‐\'’.]|[\\p{L}&&[^a-z]]', ''), '[^a-z]+', ' ')), ' '),
            SPLIT(GET(auth.authors, i).author_key, ';')[0]))
        OR (LENGTH(SPLIT(x.author_key, ';')[0]) >= 2
          AND ARRAY_CONTAINS(SPLIT(TRIM(REGEXP_REPLACE(REGEXP_REPLACE(TRANSLATE(LOWER(CONCAT_WS(' ', GET(auth.authors, i).given, GET(auth.authors, i).family, GET(auth.authors, i).name)),
            'àáâãäåçèéêëìíîïñòóôõöùúûüýÿāăąćĉċčďēĕėęěĝğġģĥĩīĭįĵķĺļľŀńņňŉōŏőŕŗřśŝşšţťũūŭůűųŵŷźżžſơưǎǐǒǔǖǘǚǜǟǡǧǩǫǭǰǵǹǻȁȃȅȇȉȋȍȏȑȓȕȗșțȟȧȩȫȭȯȱȳ',
            'aaaaaaceeeeiiiinooooouuuuyyaaaccccdeeeeegggghiiiijkllllnnnnooorrrssssttuuuuuuwyzzzsouaiouuuuuaagkoojgnaaaeeiioorruusthaeooooy'),
            '[-‐\'’.]|[\\p{L}&&[^a-z]]', ''), '[^a-z]+', ' ')), ' '),
            SPLIT(x.author_key, ';')[0])),
        FALSE)
    )) AS seats
  FROM crossref_base c
    INNER JOIN name_noise_affiliation_dois nd ON nd.doi = c.doi
    INNER JOIN crossref_authoritative_base auth ON auth.doi = c.doi AND auth.r = 1
  WHERE c.provenance = 'landing_page'
    AND c.affiliations_exist
    AND SIZE(c.authors) = SIZE(auth.authors)
    -- identical keys leave nothing for a name test: the key join already placed every string the page gives
    AND TO_JSON(TRANSFORM(c.authors, x -> x.author_key)) <> TO_JSON(TRANSFORM(auth.authors, x -> x.author_key))
),

name_noise_affiliations AS (
  SELECT doi, seat.author_position AS author_position, ARRAY_SORT(MIN_BY(seat.affiliations, r)) AS affiliations
  FROM (
    SELECT doi, r, EXPLODE(FILTER(seats, z -> SIZE(z.affiliations) > 0)) AS seat
    FROM name_noise_affiliation_seats
    WHERE FORALL(seats, z -> z.same_person)
  )
  GROUP BY doi, seat.author_position
),

-- oxjob #1793: PubMed Central open data (s3://pmc-oa-opendata, notebooks/ingest/PmcOpenData) fills what every source
-- above leaves empty, and changes nothing else: it stays out of crossref_base, so every match and fallback above runs
-- exactly as without it. An author with no affiliation takes PMC's; an author with no corresponding flag takes PMC's
-- TRUE only on DOIs where no non-PDF source flags anyone and no legacy work_authors row flags anyone on the work (a
-- source flag outranks legacy in CreateWorksBase). PMC authors join by author_key when the key is unique on both sides,
-- or by key at the same position in same-length lists; by position alone when no key matches and the lists are the
-- same length (the same guard as dois_needing_positional_fallback). Every join gives at most one PMC row per author.
-- Revert: drop the four pmc_oa CTEs, their three joins and the two COALESCE tails.
pmc_oa_flag_blocked_dois AS (
  SELECT doi FROM crossref_base
  WHERE is_corresponding_exists AND provenance != 'pdf'
    AND doi IN (SELECT doi FROM openalex.pubmed_central.pmc_oa_authors WHERE is_corresponding_exists)
  UNION
  SELECT lwi.native_id AS doi
  FROM openalex.works.location_work_ids lwi
    LEFT SEMI JOIN openalex.works_legacy.work_authors lw ON lwi.work_id = lw.work_id AND lw.is_corresponding
  -- oxjob #2275: legacy rows that flag every one of 3+ seats are no evidence (CreateWorksBase ignores them), so they no
  -- longer block PMC's flag. Revert: delete this comment and the anti join below.
    LEFT ANTI JOIN (
      SELECT work_id FROM openalex.works_legacy.work_authors
      GROUP BY work_id
      HAVING COUNT(*) >= 3 AND COUNT_IF(is_corresponding) = COUNT(*)
    ) lac ON lac.work_id = lwi.work_id
  WHERE lwi.provenance = 'crossref'
    AND lwi.native_id IN (SELECT doi FROM openalex.pubmed_central.pmc_oa_authors WHERE is_corresponding_exists)
),

pmc_oa_exploded AS (
  SELECT
    doi,
    pmc_author_count,
    author_position,
    pmc_author.author_key AS author_key,
    CASE WHEN SIZE(filter(pmc_author.affiliations, x -> x.name IS NOT NULL)) > 0
      THEN array_sort(filter(pmc_author.affiliations, x -> x.name IS NOT NULL)) END AS affiliations,
    CASE WHEN pmc_author.is_corresponding AND NOT blocked THEN TRUE END AS is_corresponding
  FROM (
    SELECT
      p.doi,
      SIZE(p.authors) AS pmc_author_count,
      b.doi IS NOT NULL AS blocked,
      posexplode(p.authors) AS (author_position, pmc_author)
    FROM openalex.pubmed_central.pmc_oa_authors p
      LEFT JOIN pmc_oa_flag_blocked_dois b ON p.doi = b.doi
    WHERE p.authors_exist
  )
),

pmc_oa_keyed AS (
  SELECT *, COUNT(*) OVER (PARTITION BY doi, author_key) AS pmc_key_count
  FROM pmc_oa_exploded
),

-- DOIs where no PMC author key matches any authoritative author key and both lists are the same length
pmc_oa_positional_dois AS (
  SELECT ba.doi
  FROM crossref_best_authors_exploded ba
    INNER JOIN (SELECT DISTINCT doi, pmc_author_count FROM pmc_oa_exploded) pc
      ON pc.doi = ba.doi AND pc.pmc_author_count = ba.author_count
    LEFT JOIN pmc_oa_keyed pk ON pk.doi = ba.doi AND pk.author_key = ba.author_key
  GROUP BY ba.doi
  HAVING COUNT(pk.author_key) = 0
),

-- DOIs where key-based affiliation matching found zero matches across all authors
-- AND author counts match between the authoritative source and best affiliation source.
-- The author count guard prevents wrong positional assignments when sources disagree on authors.
dois_needing_positional_fallback AS (
  SELECT no_keys.doi
  FROM (
    SELECT ba.doi
    FROM crossref_best_authors_exploded ba
    LEFT JOIN crossref_affiliations ca ON ba.doi = ca.doi AND ba.author_key = ca.author_key
    GROUP BY ba.doi
    HAVING COUNT(ca.author_key) = 0
  ) no_keys
  INNER JOIN (
    SELECT doi, SIZE(authors) as author_count
    FROM crossref_authoritative_base WHERE r = 1
  ) auth ON no_keys.doi = auth.doi
  INNER JOIN (
    SELECT doi, SIZE(authors) as author_count
    FROM crossref_base
    WHERE affiliations_exist
    QUALIFY ROW_NUMBER() OVER (PARTITION BY doi ORDER BY priority ASC) = 1
  ) aff ON no_keys.doi = aff.doi AND auth.author_count = aff.author_count
),

crossref_authors_and_affiliations_base AS (
  SELECT
    ba.doi,
    ba.original_author_order,
    ba.best_author_list_exploded,
    ba.author_key,
    COALESCE(
      -- oxjob #726: co-authors sharing a key take their own seat from a lined-up source, never the union
      CASE WHEN catk.doi IS NOT NULL THEN COALESCE(cat.affiliations, ARRAY()) ELSE ca.affiliations END,
      cap.affiliations, pmk.affiliations, pmp.affiliations,
      -- oxjob #1250: named entries behind an id-only first entry, only where nothing above gives any
      cna.affiliations,
      -- oxjob #2274 (fix 7): the page's affiliations at a seat its noisy name keeps off the key join, last of all
      nna.affiliations
    ) AS affiliations,
    COALESCE(
      -- a key only this author has, or co-authors sharing a key whose flags agree
      CASE WHEN ba.key_count = 1 OR cor.flags_agree THEN cor.is_corresponding_landing_page END,
      -- co-authors sharing a key flagged both ways: the flag at this author's own position
      cort.is_corresponding_landing_page,
      corp.is_corresponding_landing_page,
      -- oxjob #2223: a byline in another script, flagged by position (cross_script_corresponding above)
      xsc.is_corresponding_landing_page,
      -- oxjob #2274: the same byline with noise in its names, flagged by position (name_noise_corresponding above)
      nnc.is_corresponding_landing_page,
      -- no positional evidence: any source's TRUE, as before (minus the duplicate)
      cor.is_corresponding_landing_page,
      -- oxjob #1793: PMC open data, only where nothing above flags this author (NULL on blocked DOIs)
      pmk.is_corresponding,
      pmp.is_corresponding
    ) AS is_corresponding_landing_page
  FROM crossref_best_authors_keyed ba
    LEFT JOIN crossref_affiliations ca USING (doi, author_key)
    -- oxjob #726: shared keys settled by position
    LEFT JOIN crossref_affiliations_tiebreak_keys catk
      ON ba.key_count > 1 AND catk.doi = ba.doi AND catk.author_key = ba.author_key
    LEFT JOIN crossref_affiliations_tiebreak cat
      ON catk.doi IS NOT NULL AND cat.doi = ba.doi AND cat.author_position = ba.original_author_order
      AND cat.author_key = ba.author_key AND cat.source_author_count = ba.author_count
    LEFT JOIN crossref_is_corresponding cor USING (doi, author_key)
    LEFT JOIN crossref_is_corresponding_tiebreak cort
      ON ba.key_count > 1 AND NOT cor.flags_agree
      AND cort.doi = ba.doi AND cort.author_position = ba.original_author_order
      AND cort.author_key = ba.author_key AND cort.source_author_count = ba.author_count
    LEFT JOIN dois_needing_positional_fallback fb ON ba.doi = fb.doi
    LEFT JOIN crossref_affiliations_positional cap
      ON fb.doi IS NOT NULL AND ba.doi = cap.doi AND ba.original_author_order = cap.author_position
    LEFT JOIN crossref_is_corresponding_positional corp
      ON fb.doi IS NOT NULL AND ba.doi = corp.doi AND ba.original_author_order = corp.author_position
    LEFT JOIN cross_script_corresponding xsc
      ON xsc.doi = ba.doi AND xsc.author_position = ba.original_author_order
    LEFT JOIN name_noise_corresponding nnc
      ON nnc.doi = ba.doi AND nnc.author_position = ba.original_author_order
    LEFT JOIN pmc_oa_keyed pmk
      ON pmk.doi = ba.doi AND pmk.author_key = ba.author_key
      AND ((ba.key_count = 1 AND pmk.pmc_key_count = 1)
        OR (pmk.author_position = ba.original_author_order AND pmk.pmc_author_count = ba.author_count))
    LEFT JOIN pmc_oa_positional_dois pfb ON ba.doi = pfb.doi
    LEFT JOIN pmc_oa_exploded pmp
      ON pfb.doi IS NOT NULL AND ba.doi = pmp.doi AND ba.original_author_order = pmp.author_position
    LEFT JOIN name_noise_affiliations nna  -- oxjob #2274 (fix 7)
      ON nna.doi = ba.doi AND nna.author_position = ba.original_author_order
    -- oxjob #1250: keys only this author holds (shared keys stay with #726's tiebreak/union rules)
    LEFT JOIN crossref_named_after_id_only cna
      ON ba.key_count = 1 AND cna.doi = ba.doi AND cna.author_key = ba.author_key
),

crossref_authors_and_affiliations_staging AS (
  SELECT
    doi,
    STRUCT(
      original_author_order,
      best_author_list_exploded.given,
      best_author_list_exploded.family,
      best_author_list_exploded.name,
      best_author_list_exploded.orcid,
      affiliations,
      is_corresponding_landing_page AS is_corresponding,
      best_author_list_exploded.author_key
    ) AS authorships
  FROM crossref_authors_and_affiliations_base
),

crossref_super_authorships_built AS (
SELECT
  doi,
  TRANSFORM(
    ARRAY_SORT(
      COLLECT_SET(authorships),
      (left, right) -> CASE
        WHEN left.original_author_order < right.original_author_order THEN -1
        WHEN left.original_author_order > right.original_author_order THEN 1
        ELSE 0
      END
    ),
    x -> STRUCT(
      TRIM(x.given) AS given,
      TRIM(x.family) AS family,
      TRIM(x.name) AS name,
      x.orcid,
      TRANSFORM(
        x.affiliations,
        -- oxjob #1062: publisher pages (APS) wrap organisation names in ROR links ('<a href="https://ror.org/02en5vm52">
        -- Sorbonne Université</a>, 4 Place Jussieu ...'); the tags are dropped, the text kept. Revert: TRIM(y.name).
        y -> STRUCT(TRIM(regexp_replace(y.name, '(?i)</?a\\b[^>]*>', '')) AS name, y.department AS department, y.ror_id AS ror_id)
      ) AS affiliations,
      x.is_corresponding,
      x.author_key
    )
  ) AS authorships
FROM crossref_authors_and_affiliations_staging
GROUP BY doi
),

-- oxjob #2274: one article registered under two Crossref DOIs and merged into one work (Inderscience's pair of DOIs per
-- paper, JSTOR's 10.2307 copy of a publisher's article, an old and a new Wiley or Elsevier DOI). CreateWorksBase takes
-- the author list from one of them (the first DOI on a tie), and when that one carries no affiliations the work shows
-- none, though its sibling's page or deposit gives them (46,073 visible works, night 2026-11-15; #2274 EXPLORE § 4). A DOI
-- whose authors carry no affiliation takes them, seat for seat, from a sibling Crossref DOI on the same work (as of the
-- previous night, location_work_ids) that lists the same authors with the same keys in the same order; likewise its
-- corresponding flags when it flags no one and the sibling flags some but not every author. Preprint pairs are left to
-- #1819 (either DOI typed preprint or posted-content, or an SSRN paper). Revert: drop these three CTEs and select from
-- crossref_super_authorships_built directly.
sibling_preprint_dois AS (
  SELECT DISTINCT native_id AS doi
  FROM openalex.works.locations_parsed
  WHERE provenance = 'crossref' AND (type = 'preprint' OR raw_type = 'posted-content' OR native_id LIKE '10.2139/ssrn.%')
),

sibling_crossref_dois AS (
  SELECT DISTINCT lwi.native_id AS doi, lwi.work_id
  FROM openalex.works.location_work_ids lwi
  WHERE lwi.provenance = 'crossref'
    AND lwi.work_id IN (
      SELECT work_id FROM openalex.works.location_work_ids WHERE provenance = 'crossref'
      GROUP BY work_id HAVING COUNT(DISTINCT native_id) > 1
    )
),

sibling_fill AS (
  SELECT
    w.doi,
    -- one sibling per DOI: one with affiliations first, then one with flags, then the lowest DOI
    MIN_BY(s.authorships, NAMED_STRUCT(
      'no_aff', NOT EXISTS(s.authorships, x -> COALESCE(SIZE(x.affiliations), 0) > 0),
      'no_flag', NOT EXISTS(s.authorships, x -> COALESCE(x.is_corresponding, FALSE)),
      'doi', s.doi)) AS sibling_authorships
  FROM crossref_super_authorships_built w
    INNER JOIN sibling_crossref_dois ww ON ww.doi = w.doi
    INNER JOIN sibling_crossref_dois ws ON ws.work_id = ww.work_id AND ws.doi <> ww.doi
    INNER JOIN crossref_super_authorships_built s ON s.doi = ws.doi
    LEFT ANTI JOIN sibling_preprint_dois pw ON pw.doi = w.doi
    LEFT ANTI JOIN sibling_preprint_dois ps ON ps.doi = s.doi
  WHERE SIZE(w.authorships) = SIZE(s.authorships)
    AND TO_JSON(TRANSFORM(w.authorships, x -> x.author_key)) = TO_JSON(TRANSFORM(s.authorships, x -> x.author_key))
    AND (
      (NOT EXISTS(w.authorships, x -> COALESCE(SIZE(x.affiliations), 0) > 0)
        AND EXISTS(s.authorships, x -> COALESCE(SIZE(x.affiliations), 0) > 0))
      OR (NOT EXISTS(w.authorships, x -> COALESCE(x.is_corresponding, FALSE))
        AND EXISTS(s.authorships, x -> COALESCE(x.is_corresponding, FALSE))
        AND NOT (SIZE(s.authorships) > 1 AND FORALL(s.authorships, x -> COALESCE(x.is_corresponding, FALSE))))
    )
  GROUP BY w.doi
)

SELECT
  b.doi,
  CASE WHEN f.doi IS NULL THEN b.authorships
  ELSE ZIP_WITH(b.authorships, f.sibling_authorships, (a, s) -> STRUCT(
    a.given AS given,
    a.family AS family,
    a.name AS name,
    a.orcid AS orcid,
    CASE WHEN NOT EXISTS(b.authorships, x -> COALESCE(SIZE(x.affiliations), 0) > 0) THEN s.affiliations ELSE a.affiliations END AS affiliations,
    CASE WHEN NOT EXISTS(b.authorships, x -> COALESCE(x.is_corresponding, FALSE))
      AND NOT (SIZE(f.sibling_authorships) > 1 AND FORALL(f.sibling_authorships, x -> COALESCE(x.is_corresponding, FALSE)))
      THEN s.is_corresponding ELSE a.is_corresponding END AS is_corresponding,
    a.author_key AS author_key
  )) END AS authorships
FROM crossref_super_authorships_built b
  LEFT JOIN sibling_fill f ON f.doi = b.doi;

In [ ]:
-- oxjob #1594: initials-keyed affiliation blocks. Neurology, Stroke, Circulation, Radiology, ATVB and other journals
-- deposit ONE affiliation string for the whole paper, keyed by author initials, and repeat it under every author:
--   "From the Department of Neurology (S.M., D.R., A.K.), All India Institute of Medical Sciences, New Delhi, India;
--    Brain Health Unit (K.K., T.D.), World Health Organization, Geneva, Switzerland; ..."
-- so every author took every institution (ZD 3464: W3214050275, W4385854855, W2914576615). The block names who belongs
-- where: each author keeps only the ';'-separated segments whose initials group names them. Guards: the string sits on
-- 2+ authors of the DOI; 2+ initials groups and 2+ authors resolve; any initials token that names nobody on the byline
-- leaves the DOI exactly as it was; a token two authors share goes to neither; a block that would give every holder the
-- same segments is left as deposited; a segment naming no one stays with everyone; a truncated tail ('...') naming no
-- one is dropped, and so is the last segment of a block Crossref cut at a fixed length (1000, 399 or 250 characters) with
-- no '...' or inside a group. A group whose initials each fit two authors goes to no one, and dotted initials that name
-- no one ('(K.K.F.)') leave the DOI as deposited. Initials may use any Latin capital ('S.Č.'); a short key two
-- authors fit ('S.C.') goes to the one whose longest initials it is (Sorcha Cotter, not Stuart A. Cook). Each author's new list is a subset of the block's own text. Initials are read from both
-- given/family (this table keeps only the first given name and drops hyphens) and the display name ('Rioghna R.
-- Pittock', 'Jonathan Graff-Radford').
-- Source of truth and tests: oxjobs #1594 work/initials_split.py.txt, work/test_initials_split.py.txt (20 tests).
-- Revert: remove this cell and the next two (the next run rebuilds crossref_super_authorships without the split).
CREATE OR REPLACE FUNCTION openalex.works.pair_initials_keyed_authorships(authorships ARRAY<STRUCT<given: STRING, family: STRING, name: STRING, orcid: STRING, affiliations: ARRAY<STRUCT<name: STRING, department: STRING, ror_id: STRING>>, is_corresponding: BOOLEAN, author_key: STRING>>)
RETURNS ARRAY<STRUCT<given: STRING, family: STRING, name: STRING, orcid: STRING, affiliations: ARRAY<STRUCT<name: STRING, department: STRING, ror_id: STRING>>, is_corresponding: BOOLEAN, author_key: STRING>>
LANGUAGE PYTHON
DETERMINISTIC
COMMENT 'oxjob #1594: pair an initials-keyed affiliation block (one string under every author) with its authors; NULL leaves the DOI as deposited. Source + tests: oxjobs #1594 work/initials_split.py.txt, work/test_initials_split.py.txt'
AS $$
import html
import re
import unicodedata

PAREN_RE = re.compile(r"\(([^()]*)\)")
ITEM_SPLIT_RE = re.compile(r"\s*(?:,|\x3b|\band\b|&)\s*")
# desk 16 Nov (10.1148/ryai.220187): capitals beyond A-Z and Latin-1 ('(S.Č., N.F.)', Ł, Š, Ž) are initials too
UC = "A-ZÀ-ÖØ-Þ" + "".join(c for c in map(chr, range(0x100, 0x250)) if c.isupper())
INITIAL_ITEM_RE = re.compile(rf"^(?=.*[{UC}])(?:-?\s?(?:[{UC}][a-z]?\.?|[a-z]{{1,2}}\.)\s?){{1,8}}$")
NAME_ITEM_RE = re.compile(rf"^(?:[{UC}][\w’'-]*\.?\s+){{0,2}}(?:[a-z]+\s+){{0,2}}[{UC}][\w’'-]{{2,}}$")
NAME_SPLIT_RE = re.compile(r"[\s\-‐‑.]+")
TRUNC_RE = re.compile(r"(?:\.\.\.|…)\s*$")
# desk 16 Nov: Crossref also cuts long blocks at a fixed length with no '...' (1000, 399 or 250 characters, often
# mid-word: 'Department of Genitourinary Radiology, University of Pennsylvania Medical C'). Its last segment is then a
# fragment of a line that named someone after the cut, never a line for every author.
CUT_LENGTHS = {249, 250, 398, 399, 999, 1000}
SUFFIX_RE = re.compile(r"^(?:Jr|Sr|II|III|IV)\.?$")
APOS = "'’ʼ`"
# chr(59), the semicolon: no literal one in this function body, so no SQL statement splitter can cut the cell inside it
SEMI = chr(59)


def _letters(s):
    s = unicodedata.normalize("NFKD", s or "")
    return "".join(ch for ch in s if ch.isalpha() and not unicodedata.combining(ch)).upper()


def _initials_of(parts):
    return "".join(_letters(p)[:1] for p in parts if _letters(p))


def _family_forms(family):
    f = (family or "").strip()
    forms = set()
    plain = [p for p in NAME_SPLIT_RE.split(f) if _letters(p)]
    forms.add(_initials_of(plain))
    no_particles = [p for p in plain if not p[:1].islower()]
    if no_particles:
        forms.add(_initials_of(no_particles))
    if plain:
        forms.add(_letters(plain[0])[:1] if not plain[0][:1].islower() or len(plain) == 1 else _initials_of(no_particles)[:1])
    for ch in APOS:
        if ch in f:
            pre, post = f.split(ch, 1)
            forms.add(_initials_of([pre, post]))
            forms.add(_initials_of([post]))
            forms.add(_initials_of([pre + post]))
    return {x for x in forms if x}


def _author_variants(given, family, name=None):
    if not family and name:
        bits = [b for b in name.split() if b]
        family = bits[-1] if bits else ""
        given = " ".join(bits[:-1])
    g = [p for p in NAME_SPLIT_RE.split(given or "") if _letters(p)]
    f = [p for p in NAME_SPLIT_RE.split(family or "") if _letters(p)]
    gi, fi = _initials_of(g), _initials_of(f)
    full = gi + fi
    loose = set()
    for ff in _family_forms(family):
        if gi:
            loose |= {gi + ff, gi[:1] + ff, gi + ff[:1], gi[:1] + ff[:1]}
    loose.discard(full)
    return full, loose, _letters(family), gi[:1]


def _author_keys(given, family, name=None):
    """oxjob #1594 (desk, 16 Nov): openalex.works.crossref_super_authorships keeps only the first given name
    ('Rioghna', 'Val') and drops the family's hyphen ('GraffRadford'), while the display name keeps both
    ('Rioghna R. Pittock', 'Jonathan Graff-Radford'). Read both: (full set, loose set, family keys, first initials)."""
    vs = [_author_variants(given, family, name)]
    if name:
        vs.append(_author_variants(None, None, name))
    full = {v[0] for v in vs if v[0]}
    loose = set().union(*(v[1] for v in vs)) - full
    return full, loose, {v[2] for v in vs if v[2]}, {v[3] for v in vs if v[3]}


def _split_segments(block):
    segs, depth, cur = [], 0, []
    for ch in block:
        if ch == "(":
            depth += 1
        elif ch == ")":
            depth = max(0, depth - 1)
        if ch == SEMI and depth == 0:
            segs.append("".join(cur))
            cur = []
        else:
            cur.append(ch)
    segs.append("".join(cur))
    return [s.strip() for s in segs if s.strip()]


def _paren_items(text):
    raw = [i.strip() for i in ITEM_SPLIT_RE.split(text) if i.strip() and not SUFFIX_RE.match(i.strip())]
    if not raw:
        return None
    items = []
    for i in raw:
        if INITIAL_ITEM_RE.match(i):
            items.append(("init", _letters(i)))
        elif NAME_ITEM_RE.match(i):
            words = i.split()
            items.append(("name", _letters(words[0])[:1] if len(words) > 1 else "", _letters(words[-1])))
        else:
            return None
    return items


def _resolve(tok, authors):
    if tok[0] == "name":
        _, gi, fam = tok
        hits = [i for i, a in enumerate(authors) if fam in a[2] and (not gi or gi in a[3])]
    else:
        t = tok[1]
        hits = [i for i, a in enumerate(authors) if t in a[0]]
        if len(hits) > 1:
            # only the first given name is stored: Stuart A. Cook also reads 'SC', like Sorcha Cotter. Prefer the
            # author whose longest initials ARE the token.
            longest = [i for i in hits if len(t) == max(len(x) for x in authors[i][0])]
            if len(longest) == 1:
                hits = longest
        hits = hits or [i for i, a in enumerate(authors) if t in a[1]]
    if len(hits) == 1:
        return "ok", hits[0]
    return ("ambiguous" if hits else "unknown"), None


def _clean(seg, spans):
    out = seg
    for g in spans:
        out = out.replace(g, " ")
    out = re.sub(r"^\s*(?:(?:and\s+)?from\s+(?:the\s+)?|and\s+(?:the\s+)?)", "", out, flags=re.I)
    out = re.sub(r"^the\s+", "", out)  # "... the Institute of ..." after a semicolon (not "The College")
    out = re.sub(r"\s+,", ",", out)
    out = re.sub(r"\s{2,}", " ", out)
    out = re.sub(r"\s+(and)\s*,", r" \1", out)
    out = re.sub(r"[\s,\x3b.]*(?:\.\.\.|…)?\s*$", "", out)
    return out.strip(" ," + SEMI)


def _is_cut(block):
    b = (block or "").rstrip()
    return (bool(TRUNC_RE.search(b)) or (len(block or "") in CUT_LENGTHS and not re.search(r"[.)\]]$", b))
            or b.count("(") > b.count(")"))  # cut inside a group: '... Hong Kong, China (J.Y.S.C'


def _analyse(block, authors):
    per_author, everyone, groups = {}, [], 0
    segs = _split_segments(html.unescape(block))
    cut = _is_cut(block)
    for k, seg in enumerate(segs):
        spans, hit = [], set()
        for m in PAREN_RE.finditer(seg):
            toks = _paren_items(m.group(1))
            if toks is None:
                continue
            res = [_resolve(t, authors) for t in toks]
            if not any(r[0] == "ok" for r in res):
                if all(r[0] == "ambiguous" for r in res):
                    spans.append(m.group(0))  # initials that each fit two authors: the segment goes to no one
                elif "." in m.group(1) and all(t[0] == "init" for t in toks):
                    return None  # dotted initials naming no one ('(K.K.F.)'): the byline doesn't match, leave it alone
                continue
            if any(r[0] == "unknown" for r in res):
                return None
            groups += 1
            spans.append(m.group(0))
            hit |= {r[1] for r in res if r[0] == "ok"}
        if spans:
            c = _clean(seg, spans)
            if c and re.search(r"[A-Za-zÀ-ÿ]", c):
                for a in hit:
                    per_author.setdefault(a, [])
                    if c not in per_author[a]:
                        per_author[a].append(c)
        elif not (TRUNC_RE.search(seg) or (cut and k == len(segs) - 1)):
            c = _clean(seg, [])
            if c and re.search(r"[A-Za-zÀ-ÿ]{3}", c) and c not in everyone:
                everyone.append(c)
    if groups < 2 or len(per_author) < 2:
        return None
    return per_author, everyone


def _pair_authorships(authorships):
    """authorships: the crossref_super_authorships array. Returns the new array, or None to leave the DOI alone."""
    if not authorships:
        return None
    rows = [a.asDict(recursive=True) if hasattr(a, "asDict") else dict(a) for a in authorships]
    for r in rows:
        r["affiliations"] = [x.asDict() if hasattr(x, "asDict") else dict(x) for x in (r.get("affiliations") or []) if x is not None] if r.get("affiliations") is not None else None
    names = [[x["name"] for x in (r.get("affiliations") or []) if x and x.get("name")] for r in rows]
    counts = {}
    for n in names:
        for s in set(n):
            counts[s] = counts.get(s, 0) + 1
    shared = [s for s, c in counts.items() if c >= 2]
    if not shared:
        return None
    authors = [_author_keys(r.get("given"), r.get("family"), r.get("name")) for r in rows]
    replace = {}
    for b in shared:
        r = _analyse(b, authors)
        if r:
            holders = [i for i, n in enumerate(names) if b in n]
            lists = [sorted(r[0].get(i, []) + r[1]) for i in holders]
            if all(l == lists[0] for l in lists):
                continue  # every holder would get the same segments: one institution for all, leave it as deposited
            replace[b] = r
    if not replace:
        return None
    for i, r in enumerate(rows):
        old = {x["name"]: x for x in (r.get("affiliations") or []) if x and x.get("name")}
        out, seen = [], set()
        for s in names[i]:
            new_names = (replace[s][0].get(i, []) + replace[s][1]) if s in replace else [s]
            for n in new_names:
                if n in seen:
                    continue
                seen.add(n)
                # a segment keeps no ror_id: a ROR on the whole block names no one in particular
                out.append(old[n] if n in old else {"name": n, "department": None, "ror_id": None})
        r["affiliations"] = out
    return rows

return _pair_authorships(authorships)
$$

In [ ]:
-- oxjob #1594: DOIs whose initials-keyed block was paired with its authors. Candidates: some affiliation string sits
-- on 2+ authors AND holds 2+ parentheticals, one starting with a capital. The function pairs only a shared string with
-- 2+ resolving parenthetical groups, so this filter skips only DOIs it would leave alone; it keeps the Python function off the
-- ~8M DOIs that merely share a string (shadow 16 Nov: the looser filter took 93 min; this one 815,830 candidates, 63 s on a
-- 4X-Large, 102,139 DOIs paired, none of them outside this filter). The function returns NULL for every DOI it leaves alone: those rows stay in
-- the table with authorships NULL (readers take authorships IS NOT NULL), because a filter on the function's result
-- makes the warehouse run the function twice (once for the filter, once for the column).
-- blocks: the deposited strings the pairing replaced. CreateWorksBase reads them (cell legacy_initials_keyed_split) to pair
-- the old system's copy of the same block (works_legacy.raw_affiliation_strings), which it adds to every seat.
CREATE OR REPLACE TABLE openalex.works.crossref_initials_keyed_split AS
SELECT doi, authorships,
       ARRAY_EXCEPT(
         ARRAY_DISTINCT(FLATTEN(TRANSFORM(deposited, a -> TRANSFORM(COALESCE(a.affiliations, ARRAY()), x -> x.name)))),
         ARRAY_DISTINCT(FLATTEN(TRANSFORM(authorships, a -> TRANSFORM(COALESCE(a.affiliations, ARRAY()), x -> x.name))))) AS blocks
FROM (
  SELECT doi, authorships AS deposited, openalex.works.pair_initials_keyed_authorships(authorships) AS authorships
  FROM openalex.works.crossref_super_authorships
  -- COALESCE: an author with no affiliations (a group author) must not null out the whole DOI
  WHERE EXISTS(
    ARRAY_DISTINCT(FLATTEN(TRANSFORM(authorships, a -> TRANSFORM(COALESCE(a.affiliations, ARRAY()), x -> x.name)))),
    s -> s RLIKE '\\(\\s*\\p{Lu}[^()]*\\)'
         AND SIZE(REGEXP_EXTRACT_ALL(s, '\\([^()]*\\)', 0)) >= 2
         AND SIZE(FILTER(authorships, a -> ARRAY_CONTAINS(TRANSFORM(COALESCE(a.affiliations, ARRAY()), x -> x.name), s))) >= 2)
)

In [ ]:
-- oxjob #1594: put the paired authorships in place of the deposited ones.
MERGE INTO openalex.works.crossref_super_authorships t
USING (SELECT doi, authorships FROM openalex.works.crossref_initials_keyed_split WHERE authorships IS NOT NULL) s
ON t.doi = s.doi
WHEN MATCHED THEN UPDATE SET t.authorships = s.authorships

In [ ]:
-- oxjob #736: the corresponding author a PDF's own words name ("*Corresponding author", "Correspondence to:", "Address
-- correspondence to", an envelope or a symbol whose legend says so), read once from our stored copy of every publisher
-- PDF whose work showed none (openalex.pdf.pdf_corresponding_named: reader 0f908f5b, explicit tier only, keys only one
-- GROBID author holds, copies agreeing, four misread shapes left out; 2,400,914 DOIs, 18 Nov 2026). It fills and
-- changes nothing else: TRUE only, only on a DOI where no author is flagged after every rule above and no legacy
-- work_authors row flags anyone on its work (the same block as PMC's, #1793), only at a seat whose author_key no other
-- author of the DOI holds, never so that every seat of a 3+ author list is flagged (#2275: no evidence), and never on
-- two co-authors who share a surname. GROBID's own flag stays unused (walden 0a274d16; ~80% right, #736 EXPLORE § 1-2).
-- Bar: blind reads of a fresh random 100 gains, 97 right (#736 EXPLORE § 7). Revert: delete this cell.
MERGE INTO openalex.works.crossref_super_authorships t
USING (
  WITH named AS (
    SELECT tgt AS doi, author_keys FROM openalex.pdf.pdf_corresponding_named WHERE kind = 'doi_copy'
  ),
  legacy_flagged AS (
    SELECT DISTINCT lwi.native_id AS doi
    FROM openalex.works.location_work_ids lwi
      LEFT SEMI JOIN openalex.works_legacy.work_authors lw ON lwi.work_id = lw.work_id AND lw.is_corresponding
      LEFT ANTI JOIN (
        SELECT work_id FROM openalex.works_legacy.work_authors
        GROUP BY work_id HAVING COUNT(*) >= 3 AND COUNT_IF(is_corresponding) = COUNT(*)
      ) lac ON lac.work_id = lwi.work_id
    WHERE lwi.provenance = 'crossref' AND lwi.native_id IN (SELECT doi FROM named)
  ),
  filled AS (
    SELECT s.doi,
      TRANSFORM(s.authorships, a -> STRUCT(a.given AS given, a.family AS family, a.name AS name, a.orcid AS orcid,
        a.affiliations AS affiliations,
        CASE WHEN ARRAY_CONTAINS(n.author_keys, a.author_key)
          AND SIZE(FILTER(s.authorships, b -> b.author_key = a.author_key)) = 1 THEN TRUE ELSE a.is_corresponding END
          AS is_corresponding,
        a.author_key AS author_key)) AS authorships
    FROM openalex.works.crossref_super_authorships s
      INNER JOIN named n ON n.doi = s.doi
      LEFT ANTI JOIN legacy_flagged lf ON lf.doi = s.doi
    WHERE s.authorships IS NOT NULL
      AND NOT EXISTS(s.authorships, a -> COALESCE(a.is_corresponding, FALSE))
  )
  SELECT doi, authorships FROM filled
  WHERE EXISTS(authorships, a -> COALESCE(a.is_corresponding, FALSE))
    AND NOT (SIZE(authorships) >= 3 AND FORALL(authorships, a -> COALESCE(a.is_corresponding, FALSE)))
    -- two flagged co-authors with the same surname: the reader matches a starred name by surname, so it can't tell
    -- 'Keqiang Zhang' from 'Shuting Zhang' (bar draw 2, #736 EXPLORE § 7): leave the record alone
    AND SIZE(ARRAY_DISTINCT(TRANSFORM(FILTER(authorships, a -> COALESCE(a.is_corresponding, FALSE)),
          a -> SPLIT(a.author_key, ';')[0])))
      = SIZE(FILTER(authorships, a -> COALESCE(a.is_corresponding, FALSE)))
) u
ON t.doi = u.doi
WHEN MATCHED THEN UPDATE SET t.authorships = u.authorships

In [0]:
select * from openalex.works.crossref_super_authorships